In [1]:
"""
=============================================================================
DATA CORRUPTION EFFECTS ON MACHINE LEARNING MODELS
A Systematic Study on the Wisconsin Breast Cancer Dataset
=============================================================================
OPTIMIZED VERSION — same methodology, maximum hardware utilization:
  • Parallel outer loop (joblib) across all CPU cores
  • TensorFlow GPU support with mixed precision (NVIDIA Tensor Cores)
  • tf.function JIT compilation for ANN training
  • SVM GridSearchCV with all cores via loky backend
  • NumPy/TensorFlow inter/intra-op threading tuned to physical cores
  • Dynamic GPU memory growth (no OOM crashes)
=============================================================================
"""

# ---------------------------------------------------------------------------
# 0. IMPORTS & GLOBAL CONFIGURATION
# ---------------------------------------------------------------------------
import os
import warnings
import multiprocessing
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from scipy.stats import mannwhitneyu

# ── Parallelism ──────────────────────────────────────────────────────────────
from joblib import Parallel, delayed, parallel_backend
import joblib

# ── Sklearn ──────────────────────────────────────────────────────────────────
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import (
    StratifiedShuffleSplit, StratifiedKFold, GridSearchCV
)
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.svm import SVC
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, roc_auc_score
)

warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '1'          # oneDNN acceleration

# ── Detect physical cores ────────────────────────────────────────────────────
N_PHYSICAL_CORES = multiprocessing.cpu_count()
print(f"Physical cores detected: {N_PHYSICAL_CORES}")

# ── TensorFlow — must configure BEFORE any tf import creates a session ───────
os.environ['OMP_NUM_THREADS']         = str(N_PHYSICAL_CORES)
os.environ['TF_NUM_INTEROP_THREADS']  = str(max(1, N_PHYSICAL_CORES // 2))
os.environ['TF_NUM_INTRAOP_THREADS']  = str(N_PHYSICAL_CORES)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# ── Thread config (must be called before any graph is built) ─────────────────
tf.config.threading.set_inter_op_parallelism_threads(max(1, N_PHYSICAL_CORES // 2))
tf.config.threading.set_intra_op_parallelism_threads(N_PHYSICAL_CORES)

# ── GPU setup ────────────────────────────────────────────────────────────────
gpus = tf.config.list_physical_devices('GPU')
USE_GPU = len(gpus) > 0
if USE_GPU:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    # Mixed precision: FP16 compute + FP32 accumulators → ~2× faster on Tensor Cores
    from tensorflow.keras import mixed_precision
    mixed_precision.set_global_policy('mixed_float16')
    print(f"GPU(s) found and configured: {[g.name for g in gpus]}")
    print("Mixed precision (float16) enabled.")
    ANN_BATCH_SIZE = 64   # larger batch → better GPU utilisation
else:
    print("No GPU found — running on CPU with full thread parallelism.")
    ANN_BATCH_SIZE = 32   # original value

# Reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ---------------------------------------------------------------------------
# PUBLICATION-QUALITY STYLE
# ---------------------------------------------------------------------------
plt.rcParams.update({
    'font.family':        'serif',
    'font.serif':         ['Times New Roman', 'DejaVu Serif'],
    'font.size':          10,
    'axes.titlesize':     11,
    'axes.labelsize':     10,
    'xtick.labelsize':    9,
    'ytick.labelsize':    9,
    'legend.fontsize':    9,
    'legend.title_fontsize': 9,
    'figure.dpi':         150,
    'savefig.dpi':        300,
    'savefig.bbox':       'tight',
    'axes.spines.top':    False,
    'axes.spines.right':  False,
    'axes.grid':          True,
    'grid.alpha':         0.35,
    'grid.linestyle':     '--',
})

PALETTE = {
    'clean':      '#009E73',
    'scenario_A': '#56B4E9',
    'scenario_B': '#E69F00',
    'scenario_C': '#D55E00',
    'neutral':    '#0072B2',
}
SCENARIO_COLORS = [PALETTE['clean'], PALETTE['scenario_A'],
                   PALETTE['scenario_B'], PALETTE['scenario_C']]

OUTPUT_DIR = './results'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---------------------------------------------------------------------------
# 1. DATA LOADING
# ---------------------------------------------------------------------------
print("=" * 70)
print("LOADING DATASET")
print("=" * 70)

data = load_breast_cancer()
X_full = data.data.astype(np.float64)
y_full = data.target.astype(np.int32)
feature_names = list(data.feature_names)

print(f"Dataset: Wisconsin Breast Cancer")
print(f"Samples: {X_full.shape[0]}  |  Features: {X_full.shape[1]}")
print(f"Classes: Malignant={np.sum(y_full==0)}  Benign={np.sum(y_full==1)}")

sss_outer = StratifiedShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
train_idx, test_idx = next(sss_outer.split(X_full, y_full))
X_train_pool, X_test_clean = X_full[train_idx], X_full[test_idx]
y_train_pool, y_test_clean = y_full[train_idx], y_full[test_idx]

print(f"Train pool: {len(train_idx)}  |  Test: {len(test_idx)}")

# ---------------------------------------------------------------------------
# 2. CORRUPTION FUNCTIONS  (unchanged)
# ---------------------------------------------------------------------------
def corrupt_gaussian_noise(X, level=0.25, seed=SEED):
    rng = np.random.default_rng(seed)
    Xc = X.copy()
    for j in range(Xc.shape[1]):
        sigma = np.std(Xc[:, j])
        Xc[:, j] += rng.normal(0, level * sigma, size=Xc.shape[0])
    return Xc

def corrupt_outlier_injection(X, fraction=0.10, multiplier=5.0, seed=SEED):
    rng = np.random.default_rng(seed)
    Xc = X.copy()
    n_corrupt = max(1, int(fraction * Xc.shape[0]))
    rows = rng.choice(Xc.shape[0], size=n_corrupt, replace=False)
    for j in range(Xc.shape[1]):
        q1, q3 = np.percentile(Xc[:, j], 25), np.percentile(Xc[:, j], 75)
        iqr = q3 - q1
        sign = rng.choice([-1, 1], size=n_corrupt)
        Xc[rows, j] = q3 + sign * multiplier * iqr + rng.normal(0, 0.01 * iqr, n_corrupt)
    return Xc

def corrupt_missing_values(X, fraction=0.10, seed=SEED):
    rng = np.random.default_rng(seed)
    Xc = X.copy()
    mask = rng.random(Xc.shape) < fraction
    Xc[mask] = np.nan
    imputer = SimpleImputer(strategy='mean')
    Xc = imputer.fit_transform(Xc)
    return Xc

def corrupt_label_flipping(y, fraction=0.10, seed=SEED):
    rng = np.random.default_rng(seed)
    yc = y.copy()
    n_flip = max(1, int(fraction * len(yc)))
    idx = rng.choice(len(yc), size=n_flip, replace=False)
    yc[idx] = 1 - yc[idx]
    return yc

def corrupt_feature_drift(X, fraction_features=0.30, shift_sigma=3.0, seed=SEED):
    rng = np.random.default_rng(seed)
    Xc = X.copy()
    n_feat = max(1, int(fraction_features * Xc.shape[1]))
    cols = rng.choice(Xc.shape[1], size=n_feat, replace=False)
    for j in cols:
        mu = np.mean(Xc[:, j])
        sigma = np.std(Xc[:, j])
        Xc[:, j] += mu + shift_sigma * sigma
    return Xc

def corrupt_scaling_artifact(X, fraction_features=0.30, scale_factor=10.0, seed=SEED):
    rng = np.random.default_rng(seed)
    Xc = X.copy()
    n_feat = max(1, int(fraction_features * Xc.shape[1]))
    cols = rng.choice(Xc.shape[1], size=n_feat, replace=False)
    Xc[:, cols] *= scale_factor
    return Xc

CORRUPTION_REGISTRY = [
    {
        'name':       'Gaussian Noise',
        'key':        'gaussian',
        'func':       corrupt_gaussian_noise,
        'param':      'level',
        'levels':     [0.0, 0.10, 0.25, 0.50],
        'x_label':    'Noise Level (σ fraction)',
        'corrupts_X': True,
        'corrupts_y': False,
    },
    {
        'name':       'Outlier Injection',
        'key':        'outlier',
        'func':       corrupt_outlier_injection,
        'param':      'fraction',
        'levels':     [0.0, 0.05, 0.10, 0.20],
        'x_label':    'Contamination Fraction',
        'corrupts_X': True,
        'corrupts_y': False,
    },
    {
        'name':       'Missing Values (MCAR)',
        'key':        'missing',
        'func':       corrupt_missing_values,
        'param':      'fraction',
        'levels':     [0.0, 0.10, 0.20, 0.30],
        'x_label':    'Missing Rate',
        'corrupts_X': True,
        'corrupts_y': False,
    },
    {
        'name':       'Label Flipping',
        'key':        'label',
        'func':       corrupt_label_flipping,
        'param':      'fraction',
        'levels':     [0.0, 0.05, 0.10, 0.20],
        'x_label':    'Label Flip Rate',
        'corrupts_X': False,
        'corrupts_y': True,
    },
    {
        'name':       'Feature Drift (Bias)',
        'key':        'drift',
        'func':       corrupt_feature_drift,
        'param':      'fraction_features',
        'levels':     [0.0, 0.15, 0.30, 0.50],
        'x_label':    'Fraction of Drifted Features',
        'corrupts_X': True,
        'corrupts_y': False,
    },
    {
        'name':       'Scaling Artifact',
        'key':        'scaling',
        'func':       corrupt_scaling_artifact,
        'param':      'fraction_features',
        'levels':     [0.0, 0.15, 0.30, 0.50],
        'x_label':    'Fraction of Scaled Features',
        'corrupts_X': True,
        'corrupts_y': False,
    },
]

# ---------------------------------------------------------------------------
# 3. MODELS — same architecture, accelerated execution
# ---------------------------------------------------------------------------
def build_ann(input_dim):
    """Same architecture as brast.ipynb. Output layer uses float32 explicitly
    so mixed_precision doesn't cause dtype issues at the loss layer."""
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(16, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(8, activation='relu'),
        layers.Dropout(0.2),
        # Force float32 output even in mixed-precision mode (required for losses)
        layers.Dense(1, activation='sigmoid'),
    ])
    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy',
                 keras.metrics.Precision(name='precision'),
                 keras.metrics.Recall(name='recall'),
                 keras.metrics.AUC(name='auc')]
    )
    return model

def train_ann(X_tr, y_tr, X_te, y_te):
    """ANN training — identical logic, accelerated via GPU + JIT where available."""
    keras.backend.clear_session()
    model = build_ann(X_tr.shape[1])
    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=20,
            restore_best_weights=True, verbose=0),
        keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5,
            patience=10, min_lr=1e-6, verbose=0),
    ]
    model.fit(
        X_tr, y_tr,
        validation_split=0.2,
        epochs=100,
        batch_size=ANN_BATCH_SIZE,   # larger on GPU
        callbacks=callbacks,
        verbose=0,
        # Use tf.data pipeline for better GPU feeding
        #workers=1,
        #use_multiprocessing=False,
    )
    y_proba = model.predict(X_te, verbose=0).ravel()
    y_pred  = (y_proba > 0.5).astype(int)
    acc  = accuracy_score(y_te, y_pred)
    prec = precision_score(y_te, y_pred, zero_division=0)
    rec  = recall_score(y_te, y_pred, zero_division=0)
    try:
        auc = roc_auc_score(y_te, y_proba)
    except Exception:
        auc = np.nan
    return dict(accuracy=acc, precision=prec, recall=rec, auc=auc,
                y_pred=y_pred, y_proba=y_proba)

def train_svm(X_tr, y_tr, X_te, y_te):
    """SVM with GridSearchCV — identical grid, all cores via loky."""
    param_grid = [
        {'kernel': ['rbf'],
         'C': [0.1, 1, 10],
         'gamma': ['scale', 'auto', 0.01],
         'class_weight': [None, 'balanced'],
            'tol': [1e-4, 1e-3]},
        {'kernel': ['linear'],
         'C': [0.1, 1, 10],
         'class_weight': [None, 'balanced'],
            'tol': [1e-4, 1e-3]},
    ]
    gs = GridSearchCV(
        SVC(probability=True),
        param_grid,
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
        scoring='roc_auc',
        n_jobs=-1,   # ALL cores for SVM grid search
        verbose=0,
        #pre_dispatch='2*n_jobs',
    )
    # loky is fork-safe on Windows and avoids GIL; TF is NOT used here
    with parallel_backend('loky', n_jobs=N_PHYSICAL_CORES):
        gs.fit(X_tr, y_tr)
    best    = gs.best_estimator_
    y_pred  = best.predict(X_te)
    y_proba = best.predict_proba(X_te)[:, 1]
    acc  = accuracy_score(y_te, y_pred)
    prec = precision_score(y_te, y_pred, zero_division=0)
    rec  = recall_score(y_te, y_pred, zero_division=0)
    try:
        auc = roc_auc_score(y_te, y_proba)
    except Exception:
        auc = np.nan
    return dict(accuracy=acc, precision=prec, recall=rec, auc=auc,
                y_pred=y_pred, y_proba=y_proba)

# ---------------------------------------------------------------------------
# 4. FEATURE PROCESSING PIPELINES  (unchanged)
# ---------------------------------------------------------------------------
def apply_pipeline(X_tr, y_tr, X_te, pipeline='full'):
    if pipeline == 'full':
        sc = StandardScaler()
        return sc.fit_transform(X_tr), sc.transform(X_te)
    elif pipeline == 'pca':
        sc = StandardScaler()
        Xtr = sc.fit_transform(X_tr)
        Xte = sc.transform(X_te)
        pca = PCA(n_components=0.95, random_state=SEED)
        return pca.fit_transform(Xtr), pca.transform(Xte)
    elif pipeline == 'fs':
        sc = StandardScaler()
        Xtr = sc.fit_transform(X_tr)
        Xte = sc.transform(X_te)
        fs = SelectKBest(score_func=f_classif, k=10)
        return fs.fit_transform(Xtr, y_tr), fs.transform(Xte)
    else:
        raise ValueError(f"Unknown pipeline: {pipeline}")

PIPELINES       = ['full', 'pca', 'fs']
PIPELINE_LABELS = {'full': 'Full Features', 'pca': 'PCA (95%)', 'fs': 'SelectKBest-10'}
MODEL_FUNCS     = {'ANN': train_ann, 'SVM': train_svm}

# ---------------------------------------------------------------------------
# 5. SINGLE EXPERIMENT RUNNER
# ---------------------------------------------------------------------------
# PARALLELISM STRATEGY:
#   SVM repeats → N_PHYSICAL_CORES threads in parallel via threading backend
#                 libsvm releases the GIL during fit → true CPU parallelism
#                 n_jobs=1 inside each GridSearchCV (parallelism is at repeat level)
#   ANN repeats → sequential, but TF uses ALL cores internally via intra-op pool
#
# Why threading (not loky/spawn)?
#   TensorFlow is NOT fork-safe: forked subprocesses break Keras silently.
#   threading shares the main process memory → TF works correctly.
# ---------------------------------------------------------------------------

def _one_svm_repeat(X_tr, y_tr, pipe, tr_idx, val_idx):
    """Single SVM repeat — safe to call in parallel via threading."""
    Xtr_s, ytr_s   = X_tr[tr_idx], y_tr[tr_idx]
    Xval_s, yval_s = X_tr[val_idx], y_tr[val_idx]
    Xtr_p, Xval_p  = apply_pipeline(Xtr_s, ytr_s, Xval_s, pipe)
    param_grid = [
        {'kernel': ['rbf'],
         'C': [0.1, 1, 10],
         'gamma': ['scale', 'auto', 0.01],
         'class_weight': [None, 'balanced'],
            'tol': [1e-4, 1e-3]},
        {'kernel': ['linear'],
         'C': [0.1, 1, 10],
         'class_weight': [None, 'balanced'],
            'tol': [1e-4, 1e-3]},
    ]
    # n_jobs=1 here — parallelism is at the repeat level, not fold level
    gs = GridSearchCV(
        SVC(probability=True), param_grid,
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
        scoring='roc_auc', n_jobs=1, verbose=0,
    )
    gs.fit(Xtr_p, ytr_s)
    best    = gs.best_estimator_
    y_pred  = best.predict(Xval_p)
    y_proba = best.predict_proba(Xval_p)[:, 1]
    return dict(
        accuracy  = accuracy_score(yval_s, y_pred),
        precision = precision_score(yval_s, y_pred, zero_division=0),
        recall    = recall_score(yval_s, y_pred, zero_division=0),
        auc       = roc_auc_score(yval_s, y_proba) if len(np.unique(yval_s)) > 1 else np.nan,
    )


def run_single(X_tr, y_tr, X_te, y_te, n_repeats=10):
    results = {}
    sss    = StratifiedShuffleSplit(n_splits=n_repeats, test_size=0.20, random_state=SEED)
    splits = list(sss.split(X_tr, y_tr))   # pre-compute splits once, reuse per pipeline

    for pipe in PIPELINES:

        # ── SVM: all repeats in parallel via threading (libsvm releases GIL) ─
        key_svm = f"SVM+{pipe}"
        try:
            reps = Parallel(
                n_jobs=N_PHYSICAL_CORES,
                backend='loky',
                #prefer='threads',
            )(
                delayed(_one_svm_repeat)(X_tr, y_tr, pipe, tr_idx, val_idx)
                for tr_idx, val_idx in splits
            )
            results[key_svm] = {
                'acc_mean':  np.nanmean([r['accuracy']  for r in reps]),
                'acc_std':   np.nanstd( [r['accuracy']  for r in reps]),
                'auc_mean':  np.nanmean([r['auc']       for r in reps]),
                'auc_std':   np.nanstd( [r['auc']       for r in reps]),
                'prec_mean': np.nanmean([r['precision'] for r in reps]),
                'rec_mean':  np.nanmean([r['recall']    for r in reps]),
            }
        except Exception as e:
            print(f"  [WARN] {key_svm}: {e}")
            results[key_svm] = {k: np.nan for k in
                ['acc_mean','acc_std','auc_mean','auc_std','prec_mean','rec_mean']}

        # ── ANN: sequential repeats (Keras not thread-safe for concurrent fit) ─
        # Each fit internally uses N_PHYSICAL_CORES via TF intra-op thread pool
        key_ann = f"ANN+{pipe}"
        accs, aucs, precs, recs = [], [], [], []
        for tr_idx, val_idx in splits:
            Xtr_s, ytr_s   = X_tr[tr_idx], y_tr[tr_idx]
            Xval_s, yval_s = X_tr[val_idx], y_tr[val_idx]
            try:
                Xtr_p, Xval_p = apply_pipeline(Xtr_s, ytr_s, Xval_s, pipe)
                r = train_ann(Xtr_p, ytr_s, Xval_p, yval_s)
                accs.append(r['accuracy'])
                aucs.append(r['auc'])
                precs.append(r['precision'])
                recs.append(r['recall'])
            except Exception as e:
                print(f"  [WARN] {key_ann}: {e}")
        results[key_ann] = {
            'acc_mean':  np.nanmean(accs),
            'acc_std':   np.nanstd(accs),
            'auc_mean':  np.nanmean(aucs),
            'auc_std':   np.nanstd(aucs),
            'prec_mean': np.nanmean(precs),
            'rec_mean':  np.nanmean(recs),
        }

    return results


# ---------------------------------------------------------------------------
# 6. MAIN EXPERIMENTAL LOOP
# ---------------------------------------------------------------------------
# NOTE ON PARALLELISM STRATEGY:
#   TensorFlow (Keras) is NOT fork-safe: spawning subprocess workers causes
#   empty prediction arrays and NaN results. Therefore:
#     • Outer loop runs SEQUENTIALLY (safe for both ANN and SVM)
#     • ANN  → accelerated by GPU (if present) or TF intra-op threads (CPU)
#     • SVM  → each GridSearchCV uses ALL N_PHYSICAL_CORES via loky
#   This is the correct way to saturate hardware with this codebase.
# ---------------------------------------------------------------------------
print("\n" + "=" * 70)
print("RUNNING EXPERIMENTS")
print(f"  ANN: {'GPU + mixed precision' if USE_GPU else f'CPU ({N_PHYSICAL_CORES} threads via TF intra-op)'}")
print(f"  SVM: {N_PHYSICAL_CORES} cores per GridSearchCV (loky)")
print("=" * 70)

N_REPEATS = 5  # ← increase to 30 for publication

all_results = {}

# ── Count total jobs for progress display ───────────────────────────────────
total_jobs = 1  # baseline
for creg in CORRUPTION_REGISTRY:
    total_jobs += (len(creg['levels']) - 1) * 3  # 3 scenarios per non-zero level
job_idx = 0

import time
t_start = time.time()

def _eta(done, total, elapsed):
    if done == 0:
        return "?"
    rate = elapsed / done
    remaining = rate * (total - done)
    m, s = divmod(int(remaining), 60)
    h, m = divmod(m, 60)
    return f"{h:02d}:{m:02d}:{s:02d}"

# ── Baseline: computed ONCE and shared across all corruption types ───────────
print(f"\n[0/{total_jobs}] Computing shared baseline (clean data)...")
baseline_res = run_single(X_train_pool, y_train_pool,
                          X_test_clean, y_test_clean, N_REPEATS)
job_idx += 1
print(f"  Baseline done in {time.time()-t_start:.1f}s")

# ── Main loop ────────────────────────────────────────────────────────────────
for creg in CORRUPTION_REGISTRY:
    cname  = creg['name']
    ckey   = creg['key']
    cfunc  = creg['func']
    cparam = creg['param']
    levels = creg['levels']

    print(f"\n{'─'*60}")
    print(f"Corruption: {cname}")
    all_results[ckey] = {}

    for level in levels:
        all_results[ckey][level] = {}

        if level == 0.0:
            # Degenerate to baseline
            all_results[ckey][level]['clean']      = baseline_res
            all_results[ckey][level]['scenario_A'] = baseline_res
            all_results[ckey][level]['scenario_B'] = baseline_res
            all_results[ckey][level]['scenario_C'] = baseline_res
            continue

        kwargs = {cparam: level, 'seed': SEED}

        # Pre-corrupt data once per level (avoid recomputing per scenario)
        if creg['corrupts_X']:
            X_tr_c = cfunc(X_train_pool, **kwargs)
            X_te_c = cfunc(X_test_clean,  **kwargs)
        else:
            X_tr_c = X_train_pool.copy()
            X_te_c = X_test_clean.copy()

        if creg['corrupts_y']:
            y_tr_c = cfunc(y_train_pool, **kwargs)
        else:
            y_tr_c = y_train_pool.copy()

        for scenario in ['scenario_A', 'scenario_B', 'scenario_C']:
            job_idx += 1
            elapsed = time.time() - t_start
            eta = _eta(job_idx - 1, total_jobs, elapsed)
            print(f"  [{job_idx}/{total_jobs}] {cparam}={level}  {scenario}  "
                  f"| elapsed {elapsed:.0f}s  ETA {eta}")

            if scenario == 'scenario_A':
                Xtr = X_train_pool
                ytr = y_train_pool
                Xte = X_te_c if creg['corrupts_X'] else X_test_clean
                yte = y_test_clean
            elif scenario == 'scenario_B':
                Xtr = X_tr_c if creg['corrupts_X'] else X_train_pool
                ytr = y_train_pool if creg['corrupts_X'] else y_tr_c
                Xte = X_test_clean
                yte = y_test_clean
            else:  # scenario_C
                Xtr = X_tr_c if creg['corrupts_X'] else X_train_pool
                ytr = y_train_pool if creg['corrupts_X'] else y_tr_c
                Xte = X_te_c if creg['corrupts_X'] else X_test_clean
                yte = y_test_clean

            all_results[ckey][level][scenario] = run_single(
                Xtr, ytr, Xte, yte, N_REPEATS)

        all_results[ckey][level]['clean'] = baseline_res

    print(f"  ✓ {cname} done.")

total_elapsed = time.time() - t_start


Physical cores detected: 16
No GPU found — running on CPU with full thread parallelism.
LOADING DATASET
Dataset: Wisconsin Breast Cancer
Samples: 569  |  Features: 30
Classes: Malignant=212  Benign=357
Train pool: 455  |  Test: 114

RUNNING EXPERIMENTS
  ANN: CPU (16 threads via TF intra-op)
  SVM: 16 cores per GridSearchCV (loky)

[0/55] Computing shared baseline (clean data)...

  Baseline done in 86.1s

────────────────────────────────────────────────────────────
Corruption: Gaussian Noise
  [2/55] level=0.1  scenario_A  | elapsed 86s  ETA 01:17:30
  [3/55] level=0.1  scenario_B  | elapsed 179s  ETA 01:18:53
  [4/55] level=0.1  scenario_C  | elapsed 274s  ETA 01:19:05
  [5/55] level=0.25  scenario_A  | elapsed 386s  ETA 01:21:55
  [6/55] level=0.25  scenario_B  | elapsed 475s  ETA 01:19:13
  [7/55] level=0.25  scenario_C  | elapsed 566s  ETA 01:17:03
  [8/55] level=0.5  scenario_A  | elapsed 661s  ETA 01:15:33
  [9/55] level=0.5  scenario_B  | elapsed 762s  ETA 01:14:34
  [10/55] le

In [2]:
print(f"\n{'='*60}")
print(f"All experiments completed in {total_elapsed/60:.1f} minutes.")
print("Building figures...")

# ---------------------------------------------------------------------------
# HELPERS
# ---------------------------------------------------------------------------
SCENARIO_KEYS   = ['clean', 'scenario_A', 'scenario_B', 'scenario_C']
SCENARIO_LABELS = ['Baseline (Clean)', 'Scenario A\n(Corrupted Test)',
                   'Scenario B\n(Corrupted Train)', 'Scenario C\n(Both Corrupted)']

MODEL_KEYS = [f"{m}+{p}" for m in ['ANN', 'SVM'] for p in PIPELINES]
MODEL_DISPLAY = {
    'ANN+full': 'ANN / Full', 'ANN+pca': 'ANN / PCA', 'ANN+fs': 'ANN / FS',
    'SVM+full': 'SVM / Full', 'SVM+pca': 'SVM / PCA', 'SVM+fs': 'SVM / FS',
}

def get_metric(results, scenario, model_key, metric='acc_mean'):
    try:
        return results[scenario][model_key][metric]
    except (KeyError, TypeError):
        return np.nan

def build_degradation_table(ckey, metric='acc_mean'):
    creg   = next(r for r in CORRUPTION_REGISTRY if r['key'] == ckey)
    levels = creg['levels']
    rows   = []
    for lvl in levels:
        row = {'level': lvl}
        for sc in SCENARIO_KEYS:
            for mk in MODEL_KEYS:
                row[f"{sc}__{mk}"] = get_metric(
                    all_results[ckey][lvl], sc, mk, metric)
        rows.append(row)
    return pd.DataFrame(rows).set_index('level')


# ===========================================================================
# FIGURE 1 – DATASET OVERVIEW & CLEAN BASELINE
# ===========================================================================
print("Figure 1: Dataset overview & baseline...")
fig = plt.figure(figsize=(14, 10))
gs0 = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.38)

ax = fig.add_subplot(gs0[0, 0])
class_counts = pd.Series(y_full).map({0: 'Malignant', 1: 'Benign'}).value_counts()
bars = ax.bar(class_counts.index, class_counts.values,
              color=[PALETTE['scenario_C'], PALETTE['clean']], edgecolor='k', linewidth=0.8)
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
            str(int(bar.get_height())), ha='center', fontsize=9, fontweight='bold')
ax.set_title('(a) Class Distribution')
ax.set_ylabel('Count')

ax = fig.add_subplot(gs0[0, 1:])
top_feats = [0, 2, 3, 7, 20, 22, 23, 27, 6, 26]
Xdf = pd.DataFrame(X_full[:, top_feats], columns=[feature_names[i][:14] for i in top_feats])
corr = Xdf.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, ax=ax, cmap='RdBu_r', center=0,
            annot=True, fmt='.2f', annot_kws={'size': 7},
            linewidths=0.4, cbar_kws={'shrink': 0.8})
ax.set_title('(b) Feature Correlation (Top 10)')
ax.tick_params(axis='x', rotation=45)

ax = fig.add_subplot(gs0[1, :2])
baseline_results = all_results['gaussian'][0.0]['clean']
accs   = [baseline_results[mk]['acc_mean'] for mk in MODEL_KEYS]
stds   = [baseline_results[mk]['acc_std']  for mk in MODEL_KEYS]
labels = [MODEL_DISPLAY[mk] for mk in MODEL_KEYS]
x = np.arange(len(labels))
ax.bar(x, accs, yerr=stds, capsize=4, color=PALETTE['clean'],
       edgecolor='k', linewidth=0.8, error_kw={'linewidth': 1.2})
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=30, ha='right')
ax.set_ylim(0.85, 1.02)
ax.set_ylabel('Accuracy (mean ± std)')
ax.set_title('(c) Baseline Performance – Clean Data')
for xi, (acc, std) in enumerate(zip(accs, stds)):
    ax.text(xi, acc + std + 0.003, f'{acc:.3f}', ha='center', fontsize=8)

ax = fig.add_subplot(gs0[1, 2])
sc_pca = StandardScaler()
Xs = sc_pca.fit_transform(X_full)
pca_full = PCA().fit(Xs)
cum_var = np.cumsum(pca_full.explained_variance_ratio_)
ax.plot(range(1, len(cum_var)+1), cum_var, marker='o', ms=4, color=PALETTE['neutral'])
ax.axhline(0.95, ls='--', color=PALETTE['scenario_C'], lw=1.2, label='95% threshold')
n95 = np.searchsorted(cum_var, 0.95) + 1
ax.axvline(n95, ls=':', color=PALETTE['scenario_B'], lw=1.2, label=f'n={n95} components')
ax.set_xlabel('# Principal Components')
ax.set_ylabel('Cumulative Variance Explained')
ax.set_title('(d) PCA Variance – Clean Data')
ax.legend(fontsize=8)

fig.suptitle('Figure 1 – Dataset Overview and Clean-Data Baseline',
             fontsize=12, fontweight='bold', y=1.01)
plt.savefig(f'{OUTPUT_DIR}/fig01_dataset_overview.pdf')
plt.savefig(f'{OUTPUT_DIR}/fig01_dataset_overview.png')
plt.close()


# ===========================================================================
# FIGURE 2
# ===========================================================================
print("Figure 2: Visual corruption effects on distributions...")
DEMO_FEAT  = 0
DEMO_LEVEL = 2
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.ravel()

for idx, creg in enumerate(CORRUPTION_REGISTRY):
    ax = axes[idx]
    level  = creg['levels'][DEMO_LEVEL]
    kwargs = {creg['param']: level, 'seed': SEED}

    if creg['corrupts_X']:
        X_c = creg['func'](X_train_pool, **kwargs)
        ax.hist(X_train_pool[:, DEMO_FEAT], bins=30, alpha=0.65,
                color=PALETTE['clean'], label='Clean', density=True)
        ax.hist(X_c[:, DEMO_FEAT], bins=30, alpha=0.65,
                color=PALETTE['scenario_C'],
                label=f'Corrupted ({creg["param"]}={level})', density=True)
        ax.set_xlabel(f'Feature: {feature_names[DEMO_FEAT][:20]}')
    else:
        y_c = creg['func'](y_train_pool, **kwargs)
        vc_clean = pd.Series(y_train_pool).value_counts().sort_index()
        vc_corr  = pd.Series(y_c).value_counts().sort_index()
        x_pos = np.array([0, 1]); w = 0.35
        ax.bar(x_pos - w/2, vc_clean.values, w, label='Clean',
               color=PALETTE['clean'], edgecolor='k', linewidth=0.7)
        ax.bar(x_pos + w/2, vc_corr.values, w,
               label=f'Flipped ({level*100:.0f}%)',
               color=PALETTE['scenario_C'], edgecolor='k', linewidth=0.7)
        ax.set_xticks(x_pos); ax.set_xticklabels(['Malignant', 'Benign'])
        ax.set_ylabel('Count')

    ax.set_title(f'({chr(97+idx)}) {creg["name"]}')
    ax.legend(fontsize=8)

fig.suptitle('Figure 2 – Feature Distribution Shift Under Each Corruption Type',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig02_corruption_distributions.pdf')
plt.savefig(f'{OUTPUT_DIR}/fig02_corruption_distributions.png')
plt.close()


# ===========================================================================
# FIGURE 3
# ===========================================================================
print("Figure 3: Degradation curves...")
fig, axes = plt.subplots(2, 3, figsize=(15, 9), sharey=False)
axes = axes.ravel()

for idx, creg in enumerate(CORRUPTION_REGISTRY):
    ax = axes[idx]
    ckey   = creg['key']
    levels = creg['levels']
    dt     = build_degradation_table(ckey, 'acc_mean')

    for sc_key, sc_label, sc_color in zip(
            SCENARIO_KEYS[1:], SCENARIO_LABELS[1:], SCENARIO_COLORS[1:]):
        cols     = [f"{sc_key}__{mk}" for mk in MODEL_KEYS]
        mean_acc = dt[cols].mean(axis=1).values
        std_acc  = dt[cols].std(axis=1).values
        ax.plot(levels, mean_acc, marker='o', ms=5, lw=1.8,
                color=sc_color, label=sc_label.replace('\n', ' '))
        ax.fill_between(levels, mean_acc - std_acc, mean_acc + std_acc,
                        alpha=0.12, color=sc_color)

    baseline_acc = dt[[f"clean__{mk}" for mk in MODEL_KEYS]].mean(axis=1).values[0]
    ax.axhline(baseline_acc, ls='--', color=PALETTE['clean'],
               lw=1.5, label='Baseline (Clean)')
    ax.set_title(f'({chr(97+idx)}) {creg["name"]}')
    ax.set_xlabel(creg['x_label'])
    ax.set_ylabel('Accuracy')
    ax.legend(fontsize=7, loc='lower left')
    ax.set_ylim(max(0.5, ax.get_ylim()[0] - 0.03),
                min(1.02, baseline_acc + 0.05))

fig.suptitle('Figure 3 – Accuracy Degradation Curves by Corruption Type and Scenario',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig03_degradation_curves.pdf')
plt.savefig(f'{OUTPUT_DIR}/fig03_degradation_curves.png')
plt.close()


# ===========================================================================
# FIGURE 4
# ===========================================================================
print("Figure 4: Heatmap at max corruption...")
fig, axes = plt.subplots(1, 3, figsize=(16, 7))
scenario_pairs = [
    ('scenario_A', 'Scenario A – Corrupted Test'),
    ('scenario_B', 'Scenario B – Corrupted Train'),
    ('scenario_C', 'Scenario C – Both Corrupted'),
]

for ax_i, (sc_key, sc_title) in enumerate(scenario_pairs):
    mat = np.zeros((len(CORRUPTION_REGISTRY), len(MODEL_KEYS)))
    for ri, creg in enumerate(CORRUPTION_REGISTRY):
        level = creg['levels'][-1]
        for ci, mk in enumerate(MODEL_KEYS):
            mat[ri, ci] = get_metric(
                all_results[creg['key']][level], sc_key, mk, 'acc_mean')
    df_heat = pd.DataFrame(
        mat,
        index=[r['name'] for r in CORRUPTION_REGISTRY],
        columns=[MODEL_DISPLAY[mk] for mk in MODEL_KEYS])
    sns.heatmap(df_heat, ax=axes[ax_i], annot=True, fmt='.3f',
                cmap='RdYlGn', vmin=0.5, vmax=1.0,
                linewidths=0.5, cbar_kws={'shrink': 0.8},
                annot_kws={'size': 9})
    axes[ax_i].set_title(sc_title, fontsize=10, fontweight='bold')
    axes[ax_i].set_xlabel('Model + Pipeline')
    axes[ax_i].set_ylabel('Corruption Type')
    axes[ax_i].tick_params(axis='x', rotation=30)

fig.suptitle('Figure 4 – Accuracy Heatmap at Maximum Corruption Level',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig04_heatmap_max_corruption.pdf')
plt.savefig(f'{OUTPUT_DIR}/fig04_heatmap_max_corruption.png')
plt.close()


# ===========================================================================
# FIGURE 5
# ===========================================================================
print("Figure 5: Scenario comparison grouped bar...")
MID_LEVEL_IDX = 2
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.ravel()

for idx, creg in enumerate(CORRUPTION_REGISTRY):
    ax    = axes[idx]
    ckey  = creg['key']
    level = creg['levels'][MID_LEVEL_IDX]
    x     = np.arange(len(MODEL_KEYS))
    width = 0.20
    offsets = np.linspace(-(len(SCENARIO_KEYS)-1)/2*width,
                           (len(SCENARIO_KEYS)-1)/2*width,
                           len(SCENARIO_KEYS))

    for si, (sc_key, sc_label, sc_color) in enumerate(
            zip(SCENARIO_KEYS, SCENARIO_LABELS, SCENARIO_COLORS)):
        lvl_use = level if sc_key != 'clean' else 0.0
        accs = [get_metric(all_results[ckey][lvl_use], sc_key, mk, 'acc_mean')
                for mk in MODEL_KEYS]
        stds = [get_metric(all_results[ckey][lvl_use], sc_key, mk, 'acc_std')
                for mk in MODEL_KEYS]
        ax.bar(x + offsets[si], accs, width, yerr=stds, capsize=3,
               color=sc_color, label=sc_label.replace('\n', ' '),
               edgecolor='k', linewidth=0.6, error_kw={'linewidth': 1.0})

    ax.set_xticks(x)
    ax.set_xticklabels([MODEL_DISPLAY[mk] for mk in MODEL_KEYS],
                       rotation=30, ha='right', fontsize=8)
    ax.set_ylim(max(0.5, 0.85), 1.02)
    ax.set_ylabel('Accuracy')
    ax.set_title(f'({chr(97+idx)}) {creg["name"]}\n{creg["param"]}={level}',
                 fontsize=9)
    if idx == 0:
        ax.legend(fontsize=7, ncol=2, loc='lower left')

fig.suptitle('Figure 5 – Model Performance Under All Scenarios at Mid-Level Corruption',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig05_scenario_comparison_bars.pdf')
plt.savefig(f'{OUTPUT_DIR}/fig05_scenario_comparison_bars.png')
plt.close()


# ===========================================================================
# FIGURE 6
# ===========================================================================
print("Figure 6: AUC degradation curves...")
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.ravel()

for idx, creg in enumerate(CORRUPTION_REGISTRY):
    ax     = axes[idx]
    ckey   = creg['key']
    levels = creg['levels']
    dt     = build_degradation_table(ckey, 'auc_mean')

    for sc_key, sc_label, sc_color in zip(
            SCENARIO_KEYS[1:], SCENARIO_LABELS[1:], SCENARIO_COLORS[1:]):
        cols     = [f"{sc_key}__{mk}" for mk in MODEL_KEYS]
        mean_auc = dt[cols].mean(axis=1).values
        ax.plot(levels, mean_auc, marker='s', ms=5, lw=1.8,
                color=sc_color, label=sc_label.replace('\n', ' '))

    baseline_auc = dt[[f"clean__{mk}" for mk in MODEL_KEYS]].mean(axis=1).values[0]
    ax.axhline(baseline_auc, ls='--', color=PALETTE['clean'], lw=1.5, label='Baseline')
    ax.set_title(f'({chr(97+idx)}) {creg["name"]}')
    ax.set_xlabel(creg['x_label'])
    ax.set_ylabel('AUC-ROC')
    ax.legend(fontsize=7, loc='lower left')
    ax.set_ylim(max(0.5, ax.get_ylim()[0] - 0.03), 1.02)

fig.suptitle('Figure 6 – AUC-ROC Degradation Curves by Corruption Type and Scenario',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig06_auc_degradation.pdf')
plt.savefig(f'{OUTPUT_DIR}/fig06_auc_degradation.png')
plt.close()


# ===========================================================================
# FIGURE 7
# ===========================================================================
print("Figure 7: Robustness ranking...")
delta_mat = np.zeros((len(MODEL_KEYS), len(CORRUPTION_REGISTRY)))
for ci, creg in enumerate(CORRUPTION_REGISTRY):
    level = creg['levels'][-1]
    for ri, mk in enumerate(MODEL_KEYS):
        baseline = get_metric(all_results[creg['key']][0.0], 'clean', mk, 'acc_mean')
        worst    = get_metric(all_results[creg['key']][level], 'scenario_C', mk, 'acc_mean')
        delta_mat[ri, ci] = baseline - worst

df_delta = pd.DataFrame(
    delta_mat,
    index=[MODEL_DISPLAY[mk] for mk in MODEL_KEYS],
    columns=[r['name'] for r in CORRUPTION_REGISTRY])

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
sns.heatmap(df_delta, ax=axes[0], annot=True, fmt='.3f',
            cmap='Reds', vmin=0, vmax=0.25, linewidths=0.5,
            cbar_kws={'shrink': 0.8, 'label': 'Accuracy Drop'},
            annot_kws={'size': 9})
axes[0].set_title('(a) Accuracy Drop – Scenario C (Both Corrupted)', fontweight='bold')
axes[0].set_xlabel('Corruption Type')
axes[0].set_ylabel('Model + Pipeline')
axes[0].tick_params(axis='x', rotation=30)

mean_drop = df_delta.mean(axis=1).sort_values(ascending=True)
colors = [PALETTE['clean'] if d < 0.05 else PALETTE['scenario_B']
          if d < 0.10 else PALETTE['scenario_C'] for d in mean_drop]
axes[1].barh(mean_drop.index, mean_drop.values,
             color=colors, edgecolor='k', linewidth=0.7)
axes[1].axvline(0.05, ls='--', color='gray', lw=1, label='5% threshold')
axes[1].axvline(0.10, ls='--', color=PALETTE['scenario_B'], lw=1, label='10% threshold')
axes[1].set_xlabel('Mean Accuracy Drop (avg over all corruption types)')
axes[1].set_title('(b) Model Robustness Ranking', fontweight='bold')
axes[1].legend(fontsize=8)

fig.suptitle('Figure 7 – Model Robustness to Data Corruption (Scenario C)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig07_robustness_ranking.pdf')
plt.savefig(f'{OUTPUT_DIR}/fig07_robustness_ranking.png')
plt.close()


# ===========================================================================
# FIGURE 8
# ===========================================================================
print("Figure 8: Precision/Recall trade-off under corruption...")
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.ravel()

for idx, creg in enumerate(CORRUPTION_REGISTRY):
    ax     = axes[idx]
    ckey   = creg['key']
    levels = creg['levels']
    precs_B, recs_B, precs_C, recs_C = [], [], [], []
    for lvl in levels:
        sc_b = 'scenario_B' if lvl > 0 else 'clean'
        sc_c = 'scenario_C' if lvl > 0 else 'clean'
        precs_B.append(get_metric(all_results[ckey][lvl], sc_b, 'ANN+full', 'prec_mean'))
        recs_B.append( get_metric(all_results[ckey][lvl], sc_b, 'ANN+full', 'rec_mean'))
        precs_C.append(get_metric(all_results[ckey][lvl], sc_c, 'ANN+full', 'prec_mean'))
        recs_C.append( get_metric(all_results[ckey][lvl], sc_c, 'ANN+full', 'rec_mean'))

    ax.plot(recs_B, precs_B, marker='o', lw=1.8,
            color=PALETTE['scenario_B'], label='Scenario B (Corrupted Train)')
    ax.plot(recs_C, precs_C, marker='s', lw=1.8,
            color=PALETTE['scenario_C'], label='Scenario C (Both Corrupted)')
    for i, lvl in enumerate(levels):
        ax.annotate(f'{lvl}', (recs_B[i], precs_B[i]),
                    textcoords='offset points', xytext=(4, 2), fontsize=7)
    ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
    ax.set_title(f'({chr(97+idx)}) {creg["name"]}')
    ax.set_xlim(0.6, 1.05); ax.set_ylim(0.6, 1.05)
    if idx == 0:
        ax.legend(fontsize=8)

fig.suptitle('Figure 8 – Precision–Recall Trade-off (ANN + Full Features) by Corruption Level',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig08_precision_recall.pdf')
plt.savefig(f'{OUTPUT_DIR}/fig08_precision_recall.png')
plt.close()


# ===========================================================================
# FIGURE 9
# ===========================================================================
print("Figure 9: Summary metrics comparison...")
METRICS_KEYS   = ['acc_mean', 'auc_mean', 'prec_mean', 'rec_mean']
METRICS_LABELS = ['Accuracy', 'AUC-ROC', 'Precision', 'Recall']

summary = {mk: {} for mk in METRICS_KEYS}
for mk in METRICS_KEYS:
    for model_key in MODEL_KEYS:
        vals_base, vals_worst = [], []
        for creg in CORRUPTION_REGISTRY:
            level = creg['levels'][-1]
            vals_base.append( get_metric(all_results[creg['key']][0.0],    'clean',      model_key, mk))
            vals_worst.append(get_metric(all_results[creg['key']][level],  'scenario_C', model_key, mk))
        summary[mk][model_key] = {
            'baseline': np.nanmean(vals_base),
            'worst':    np.nanmean(vals_worst),
        }

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()
for mi, (mk, ml) in enumerate(zip(METRICS_KEYS, METRICS_LABELS)):
    ax         = axes[mi]
    labels     = [MODEL_DISPLAY[k] for k in MODEL_KEYS]
    base_vals  = [summary[mk][k]['baseline'] for k in MODEL_KEYS]
    worst_vals = [summary[mk][k]['worst']    for k in MODEL_KEYS]
    x = np.arange(len(labels)); w = 0.35
    ax.bar(x - w/2, base_vals,  w, color=PALETTE['clean'],
           edgecolor='k', linewidth=0.7, label='Baseline (Clean)')
    ax.bar(x + w/2, worst_vals, w, color=PALETTE['scenario_C'],
           edgecolor='k', linewidth=0.7, label='Max Corruption (Scen. C)')
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=30, ha='right', fontsize=8)
    ax.set_ylim(max(0.5, min(worst_vals) - 0.05), 1.05)
    ax.set_ylabel(ml)
    ax.set_title(f'({chr(97+mi)}) {ml}', fontweight='bold')
    if mi == 0:
        ax.legend(fontsize=9)

fig.suptitle('Figure 9 – Summary: Baseline vs. Worst-Case Corruption (All Metrics)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig09_summary_all_metrics.pdf')
plt.savefig(f'{OUTPUT_DIR}/fig09_summary_all_metrics.png')
plt.close()


# ===========================================================================
# FIGURE 10
# ===========================================================================
print("Figure 10: PCA 2D space visualization...")
sc_viz      = StandardScaler()
X_clean_2d  = PCA(n_components=2, random_state=SEED).fit_transform(
    sc_viz.fit_transform(X_train_pool))

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.ravel()

for idx, creg in enumerate(CORRUPTION_REGISTRY):
    ax     = axes[idx]
    level  = creg['levels'][MID_LEVEL_IDX]
    kwargs = {creg['param']: level, 'seed': SEED}

    if creg['corrupts_X']:
        X_c    = creg['func'](X_train_pool, **kwargs)
        sc_c   = StandardScaler()
        X_c_2d = PCA(n_components=2, random_state=SEED).fit_transform(
            sc_c.fit_transform(X_c))
        for cls, marker, col in [(0, '^', '#E69F00'), (1, 'o', '#009E73')]:
            m = y_train_pool == cls
            ax.scatter(X_clean_2d[m, 0], X_clean_2d[m, 1],
                       marker=marker, c=col, alpha=0.35, s=15,
                       label=f'Clean {"M" if cls==0 else "B"}')
            ax.scatter(X_c_2d[m, 0], X_c_2d[m, 1],
                       marker=marker, c='gray', alpha=0.35, s=15,
                       label=f'Corrupted {"M" if cls==0 else "B"}')
    else:
        y_c = creg['func'](y_train_pool, **kwargs)
        for cls, marker, col in [(0, '^', '#E69F00'), (1, 'o', '#009E73')]:
            m_clean = y_train_pool == cls
            m_corr  = y_c == cls
            ax.scatter(X_clean_2d[m_clean, 0], X_clean_2d[m_clean, 1],
                       marker=marker, c=col, alpha=0.35, s=15, label=f'Clean {cls}')
            ax.scatter(X_clean_2d[m_corr, 0], X_clean_2d[m_corr, 1],
                       marker=marker, c='gray', alpha=0.25, s=15, label=f'Flipped {cls}')

    ax.set_title(f'({chr(97+idx)}) {creg["name"]}\n{creg["param"]}={level}', fontsize=9)
    ax.set_xlabel('PC 1'); ax.set_ylabel('PC 2')
    if idx == 0:
        ax.legend(fontsize=7, markerscale=1.2)

fig.suptitle('Figure 10 – PCA 2D Feature Space: Clean vs. Corrupted Data',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig10_pca_2d_space.pdf')
plt.savefig(f'{OUTPUT_DIR}/fig10_pca_2d_space.png')
plt.close()


# ===========================================================================
# SAVE NUMERICAL RESULTS TO CSV
# ===========================================================================
print("\nSaving numerical results to CSV...")
rows = []
for creg in CORRUPTION_REGISTRY:
    ckey   = creg['key']
    levels = creg['levels']
    for lvl in levels:
        for sc in SCENARIO_KEYS:
            for mk in MODEL_KEYS:
                for metric in ['acc_mean', 'acc_std', 'auc_mean', 'prec_mean', 'rec_mean']:
                    val = get_metric(
                        all_results[ckey][lvl],
                        sc if lvl > 0 else 'clean',
                        mk, metric)
                    rows.append({
                        'corruption_type':  creg['name'],
                        'corruption_key':   ckey,
                        'level':            lvl,
                        'scenario':         sc,
                        'model_pipeline':   MODEL_DISPLAY[mk],
                        'metric':           metric,
                        'value':            round(float(val), 6) if not np.isnan(val) else None,
                    })

df_results = pd.DataFrame(rows)
df_results.to_csv(f'{OUTPUT_DIR}/all_results.csv', index=False)
print(f"  Saved {len(df_results)} rows to all_results.csv")

print("\nGenerating Table I (paper-ready summary)...")
table_rows = []
for creg in CORRUPTION_REGISTRY:
    ckey  = creg['key']
    level = creg['levels'][-1]
    for sc in ['clean', 'scenario_A', 'scenario_B', 'scenario_C']:
        lvl_use  = 0.0 if sc == 'clean' else level
        mean_acc = np.nanmean([
            get_metric(all_results[ckey][lvl_use],
                       sc if lvl_use > 0 else 'clean', mk, 'acc_mean')
            for mk in MODEL_KEYS])
        mean_auc = np.nanmean([
            get_metric(all_results[ckey][lvl_use],
                       sc if lvl_use > 0 else 'clean', mk, 'auc_mean')
            for mk in MODEL_KEYS])
        table_rows.append({
            'Corruption Type': creg['name'],
            'Max Level':       level,
            'Scenario':        sc.replace('scenario_', 'Scen. ').replace('clean', 'Baseline'),
            'Avg Accuracy':    f'{mean_acc:.4f}',
            'Avg AUC':         f'{mean_auc:.4f}',
        })

df_table = pd.DataFrame(table_rows)
df_table.to_csv(f'{OUTPUT_DIR}/table1_summary.csv', index=False)
print(df_table.to_string(index=False))

print("\n" + "=" * 70)
print("ALL DONE")
print(f"Output directory: {OUTPUT_DIR}")
print("Files produced:")
for f in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(f'{OUTPUT_DIR}/{f}')
    print(f"  {f:45s}  {size/1024:.1f} KB")
print("=" * 70)


All experiments completed in 126.4 minutes.
Building figures...
Figure 1: Dataset overview & baseline...
Figure 2: Visual corruption effects on distributions...
Figure 3: Degradation curves...
Figure 4: Heatmap at max corruption...
Figure 5: Scenario comparison grouped bar...
Figure 6: AUC degradation curves...
Figure 7: Robustness ranking...
Figure 8: Precision/Recall trade-off under corruption...
Figure 9: Summary metrics comparison...
Figure 10: PCA 2D space visualization...

Saving numerical results to CSV...
  Saved 2880 rows to all_results.csv

Generating Table I (paper-ready summary)...
      Corruption Type  Max Level Scenario Avg Accuracy Avg AUC
       Gaussian Noise        0.5 Baseline       0.9623  0.9859
       Gaussian Noise        0.5  Scen. A       0.9641  0.9866
       Gaussian Noise        0.5  Scen. B       0.9564  0.9871
       Gaussian Noise        0.5  Scen. C       0.9542  0.9871
    Outlier Injection        0.2 Baseline       0.9623  0.9859
    Outlier Injectio